# Google - Cloud Service Customer Retention and Cost Efficiency

In [1]:
import pandas as pd   
import numpy as np  
import polars as pl
from datetime import date

In [2]:
df_transaction = pd.read_csv('../Data/007/fct_transactions_2.csv', parse_dates=['transaction_date'])

pl_transaction = pl.read_csv('../Data/007/fct_transactions_2.csv', try_parse_dates=True)

# Pregunta 1

### Queremos evaluar la adopción temprana del servicio premium entre los clientes empresariales. ¿Cuántos clientes únicos, cuyos códigos de nivel de servicio comienzan con 'PREM', completaron transacciones desde el 1 de abril hasta el 30 de abril de 2024?

```SQL
SELECT
    DISTINCT customer_id
FROM fct_transactions
WHERE (service_tier_code LIKE 'PREM%') AND
      ((EXTRACT (MONTH FROM transaction_date) = 4) AND
       (EXTRACT (YEAR FROM transaction_date) = 2024 ))
```

In [16]:
abril = df_transaction[
    (df_transaction['service_tier_code'].str.startswith('PREM')) &
    (df_transaction['transaction_date'].dt.month == 4) &
    (df_transaction['transaction_date'].dt.year == 2024)
].copy()

res = abril[['customer_id']].drop_duplicates().reset_index(drop=True)

In [19]:
res = pl_transaction.filter(
    (pl.col('service_tier_code').str.starts_with('PREM')) &
    (pl.col('transaction_date').dt.month() == 4) &
    (pl.col('transaction_date').dt.year() == 2024)
).select('customer_id').unique()

# Pregunta 2

### Necesitamos entender las tendencias de uso de los diferentes niveles de servicio para perfeccionar nuestros paquetes de servicios. ¿Qué códigos de nivel de servicio fueron utilizados con mayor frecuencia por los clientes para realizar transacciones en mayo de 2024, ordenados de mayor a menor frecuencia?

```SQL
SELECT
    service_tier_code,
    COUNT(transaction_id) AS total_transaction
FROM fct_transactions
WHERE ((EXTRACT(MONTH FROM transaction_date) = 5) AND
      (EXTRACT(YEAR FROM transaction_date) = 2024))
GROUP BY service_tier_code
ORDER BY total_transaction DESC;
```

In [30]:
mayo = df_transaction[
    (df_transaction['transaction_date'].dt.month == 5) &
    (df_transaction['transaction_date'].dt.year == 2024)
].reset_index()

res = mayo.groupby('service_tier_code').agg(
    total_transaction = ('transaction_id','count')
)

res = res.sort_values('total_transaction', ascending = False).reset_index()

In [31]:
res = pl_transaction.filter(
    (pl.col('transaction_date').dt.month() == 5) &
    (pl.col('transaction_date').dt.year() == 2024)
).group_by('service_tier_code').agg(
    pl.len().alias('total_transaction')
).sort('total_transaction', descending=True)


# Pregunta 3

### Queremos identificar con precisión los niveles de servicio más activos para informar los ajustes de precios en nuestras ofertas de la nube para empresas. Para las transacciones realizadas entre el 1 y el 30 de junio de 2024, ¿cuáles son los tres niveles de servicio principales según el volumen de transacciones y cuántas transacciones se registraron para cada uno?

```SQL
SELECT
    service_tier_code,
    COUNT(transaction_id) AS total_transaction
FROM fct_transactions
WHERE ((EXTRACT(MONTH FROM transaction_date) = 6) AND
       (EXTRACT(YEAR FROM transaction_date) = 2024))
GROUP BY service_tier_code
ORDER BY total_transaction DESC
LIMIT 3;
```

In [34]:
junio = df_transaction[
    (df_transaction['transaction_date'].dt.month == 6) &
    (df_transaction['transaction_date'].dt.year == 2024)
].reset_index()

res = junio.groupby('service_tier_code').agg(
    total_transaction = ('transaction_id', 'count')
).copy()

res = res.sort_values('total_transaction', ascending=False).head(3)

In [37]:
res = pl_transaction.filter(
    (pl.col('transaction_date').dt.month() == 6) &
    (pl.col('transaction_date').dt.year() == 2024)
).group_by('service_tier_code').agg(
    pl.len().alias('total_transaction')
).sort('total_transaction', descending=True).head(3)

res

service_tier_code,total_transaction
str,u32
"""PREM_BASIC""",3
"""STANDARD""",3
"""PREM_PLUS""",2
